http://localhost:8890/tree

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

# Cargar dataset
df = pd.read_csv('network_traffic.csv')
print(f"Shape: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()

/home/maykol/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Shape: (10000, 10)

Columnas: ['timestamp', 'src_ip', 'dst_ip', 'dst_port', 'protocol', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'label']

Primeras filas:


,timestamp,src_ip,dst_ip,dst_port,protocol,bytes_sent,bytes_recv,duration_sec,packets,label
0,2024-05-26 04:28:31,10.0.2.118,201.250.32.133,21,TCP,15310,45067,42.01,16,normal
1,2024-05-11 01:38:15,10.0.1.189,193.199.92.89,21,UDP,3692990,68116,55.35,2823,normal
2,2024-05-17 21:00:41,10.0.3.254,202.222.194.7,443,ICMP,23007,950555,6.64,18,normal
3,2024-05-09 18:30:38,10.0.1.254,152.183.142.33,25,TCP,467423,56148,3.12,656,normal
4,2024-05-27 17:50:29,10.0.3.87,64.39.138.32,3306,UDP,15355,18895,16.82,23,normal


In [2]:
print("=== ANÁLISIS EXPLORATORIO ===\n")
print(f"Total registros : {len(df)}")
print(f"Valores nulos   :\n{df.isnull().sum()}")
print(f"\nDistribución de etiquetas:")
print(df['label'].value_counts())
print(f"\nEstadísticas numéricas:")
df.describe()

=== ANÁLISIS EXPLORATORIO ===

Total registros : 10000
Valores nulos   :
timestamp       0
src_ip          0
dst_ip          0
dst_port        0
protocol        0
bytes_sent      0
bytes_recv      0
duration_sec    0
packets         0
label           0
dtype: int64

Distribución de etiquetas:
label
normal     9500
anomaly     500
Name: count, dtype: int64

Estadísticas numéricas:


,dst_port,bytes_sent,bytes_recv,duration_sec,packets
count,10000.000000,1.000000e+04,1.000000e+04,10000.000000,1.000000e+04
mean,5272.963700,2.815289e+07,4.124360e+05,447.154662,1.605501e+04
std,7348.395782,3.115671e+08,1.964278e+06,4530.488171,1.672859e+05
min,21.000000,1.500000e+01,0.000000e+00,0.000000,1.000000e+00
25%,53.000000,5.544000e+03,1.328800e+04,8.507500,5.000000e+00
50%,3389.000000,2.233900e+04,5.529050e+04,21.435000,2.400000e+01
75%,8080.000000,9.478175e+04,2.213258e+05,44.145000,1.100000e+02
max,65460.000000,4.987050e+09,8.155783e+07,83028.150000,2.939448e+06


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribución de bytes enviados
axes[0,0].hist(df['bytes_sent'], bins=50, color='steelblue', edgecolor='black')
axes[0,0].set_title('Distribución de Bytes Enviados')
axes[0,0].set_xlabel('bytes_sent')

# Distribución de duración
axes[0,1].hist(df['duration_sec'], bins=50, color='coral', edgecolor='black')
axes[0,1].set_title('Distribución de Duración (seg)')
axes[0,1].set_xlabel('duration_sec')

# Protocolos
proto_counts = df['protocol'].value_counts()
axes[1,0].bar(proto_counts.index, proto_counts.values, color='mediumseagreen')
axes[1,0].set_title('Distribución por Protocolo')

# Etiquetas
label_counts = df['label'].value_counts()
axes[1,1].pie(label_counts.values, labels=label_counts.index,
              autopct='%1.1f%%', colors=['steelblue','crimson'])
axes[1,1].set_title('Normal vs Anomalía')

plt.suptitle('EDA - Network Traffic Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_exploratorio.png', dpi=150)
plt.show()
print("[+] Guardado: eda_exploratorio.png")

[+] Guardado: eda_exploratorio.png


In [4]:
print("=== FEATURE ENGINEERING ===\n")

# Codificar protocolo
df['protocol_enc'] = df['protocol'].map({'TCP': 0, 'UDP': 1, 'ICMP': 2}).fillna(3)

# Nuevas features
df['bytes_ratio']    = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_per_pkt']  = df['bytes_sent'] / (df['packets'] + 1)
df['pkts_per_sec']   = df['packets']    / (df['duration_sec'] + 1)

# Features finales para el modelo
FEATURES = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets',
            'protocol_enc', 'bytes_ratio', 'bytes_per_pkt', 'pkts_per_sec',
            'dst_port']

X = df[FEATURES].copy()

# Normalización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features seleccionadas : {FEATURES}")
print(f"Shape después de escalar: {X_scaled.shape}")
print("\nPrimeras 3 filas normalizadas:")
print(X_scaled[:3])

=== FEATURE ENGINEERING ===

Features seleccionadas : ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'protocol_enc', 'bytes_ratio', 'bytes_per_pkt', 'pkts_per_sec', 'dst_port']
Shape después de escalar: (10000, 9)

Primeras 3 filas normalizadas:
[[-0.09031437 -0.18703428 -0.08943074 -0.09588265 -0.61588497 -0.03561472
   0.00362558 -0.09287552 -0.71474462]
 [-0.07850997 -0.17529961 -0.0864861  -0.07910215  1.15848832 -0.03515916
   0.80570694 -0.03044345 -0.71474462]
 [-0.09028966  0.27396626 -0.09723824 -0.09587069  2.93286162 -0.03561739
   0.61495961 -0.09038453 -0.65731424]]


In [5]:
print("=== ENTRENAMIENTO: ISOLATION FOREST ===\n")

# Etiquetas reales (1=normal, -1=anomalía)
y_true = np.where(df['label'] == 'normal', 1, -1)

# Entrenar modelo
modelo = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42
)
modelo.fit(X_scaled)

# Predicciones
y_pred = modelo.predict(X_scaled)
scores  = modelo.decision_function(X_scaled)

print(f"Total anomalías detectadas : {(y_pred == -1).sum()}")
print(f"Total normales detectados  : {(y_pred == 1).sum()}")

=== ENTRENAMIENTO: ISOLATION FOREST ===

Total anomalías detectadas : 500
Total normales detectados  : 9500


In [6]:
print("=== MÉTRICAS DEL MODELO ===\n")

precision = precision_score(y_true, y_pred, pos_label=-1)
recall    = recall_score(y_true, y_pred, pos_label=-1)
f1        = f1_score(y_true, y_pred, pos_label=-1)
cm        = confusion_matrix(y_true, y_pred)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"\nMatriz de Confusión:\n{cm}")
print(f"\nReporte completo:")
print(classification_report(y_true, y_pred, target_names=['Anomalía','Normal']))

# Gráfica de la matriz de confusión
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Anomalía','Normal'],
            yticklabels=['Anomalía','Normal'], ax=ax)
ax.set_title('Matriz de Confusión - Isolation Forest')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Predicción')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150)
plt.show()
print("[+] Guardado: matriz_confusion.png")


=== MÉTRICAS DEL MODELO ===

Precision : 0.7580
Recall    : 0.7580
F1-Score  : 0.7580

Matriz de Confusión:
[[ 379  121]
 [ 121 9379]]

Reporte completo:
              precision    recall  f1-score   support

    Anomalía       0.76      0.76      0.76       500
      Normal       0.99      0.99      0.99      9500

    accuracy                           0.98     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.98      0.98      0.98     10000

[+] Guardado: matriz_confusion.png


In [7]:
print("=== BÚSQUEDA DE UMBRAL ÓPTIMO F1 ===\n")

umbrales = np.linspace(scores.min(), scores.max(), 200)
f1_scores = []

for umbral in umbrales:
    y_pred_u = np.where(scores < umbral, -1, 1)
    f1_u = f1_score(y_true, y_pred_u, pos_label=-1, zero_division=0)
    f1_scores.append(f1_u)

idx_optimo  = np.argmax(f1_scores)
umbral_opt  = umbrales[idx_optimo]
f1_optimo   = f1_scores[idx_optimo]

print(f"Umbral óptimo : {umbral_opt:.6f}")
print(f"F1 óptimo     : {f1_optimo:.4f}")

# Gráfica umbral vs F1
plt.figure(figsize=(10,4))
plt.plot(umbrales, f1_scores, color='darkorange', linewidth=2)
plt.axvline(umbral_opt, color='red', linestyle='--', label=f'Umbral óptimo={umbral_opt:.4f}')
plt.xlabel('Umbral')
plt.ylabel('F1-Score')
plt.title('F1-Score vs Umbral de decisión')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('umbral_optimo.png', dpi=150)
plt.show()
print("[+] Guardado: umbral_optimo.png")

=== BÚSQUEDA DE UMBRAL ÓPTIMO F1 ===

Umbral óptimo : -0.001028
F1 óptimo     : 0.7578
[+] Guardado: umbral_optimo.png


In [8]:
print("=== TOP 10 ANOMALÍAS DETECTADAS ===\n")

df['anomaly_score'] = scores
df['prediccion']    = np.where(y_pred == -1, 'anomalia', 'normal')

top10_anomalias = (df[df['prediccion'] == 'anomalia']
                   .sort_values('anomaly_score')
                   .head(10)[['timestamp','src_ip','dst_ip','dst_port',
                               'protocol','bytes_sent','bytes_recv',
                               'duration_sec','packets','anomaly_score']])

print(top10_anomalias.to_string(index=False))
top10_anomalias.to_csv('top10_anomalias.csv', index=False)
print("\n[+] Guardado: top10_anomalias.csv")

=== TOP 10 ANOMALÍAS DETECTADAS ===

          timestamp     src_ip          dst_ip  dst_port protocol  bytes_sent  bytes_recv  duration_sec  packets  anomaly_score
2024-05-21 04:32:45 10.0.3.174  185.220.101.45       443      TCP  4553566747        1016       2997.88   636658      -0.317404
2024-05-07 02:28:18  10.0.3.75 143.109.217.176      8080     ICMP  4006296316       12306       3330.27   344980      -0.310577
2024-05-03 05:52:03 10.0.1.180     55.56.35.72       443      TCP  4626019978       28922       2749.17   489468      -0.308597
2024-05-10 05:42:02  10.0.2.73  185.220.101.45        53      TCP  4964770492       30659       2835.25   696949      -0.308049
2024-05-27 05:52:38  10.0.3.77    181.53.80.40        53      TCP  4921794252       22881       2306.58   812493      -0.306956
2024-05-26 03:40:51  10.0.1.54   194.165.16.89      8080      UDP  4419016764       27293       1854.90   688461      -0.305572
2024-05-17 04:39:05 10.0.1.254   38.168.189.92        80      UDP  

In [9]:
print("=== EXPORTANDO MODELO Y SCALER ===\n")

with open('modelo_anomalias.pkl', 'wb') as f:
    pickle.dump({'modelo': modelo, 'scaler': scaler,
                 'features': FEATURES, 'umbral': umbral_opt}, f)

print("[+] Modelo guardado: modelo_anomalias.pkl")
print(f"    Features  : {FEATURES}")
print(f"    Umbral    : {umbral_opt:.6f}")
print(f"    F1 óptimo : {f1_optimo:.4f}")

=== EXPORTANDO MODELO Y SCALER ===

[+] Modelo guardado: modelo_anomalias.pkl
    Features  : ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'protocol_enc', 'bytes_ratio', 'bytes_per_pkt', 'pkts_per_sec', 'dst_port']
    Umbral    : -0.001028
    F1 óptimo : 0.7578
